# Publish only the cleaned OSS 4008 student folder to GitHub (Notebook 02)

Copies **only** the current README, source code, tests, three notebooks and screenshots from the Google Drive project.
It removes superseded files **only inside `students/hamza_arab/`**, never other students.
Use a hidden GitHub personal access token prompt; never save credentials in notebook cells.


In [ ]:
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build

auth.authenticate_user()
creds, _ = default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
drive = build("drive", "v3", credentials=creds)
print("Google Drive access ready")

In [ ]:
from pathlib import Path
from io import BytesIO
import json, re, shutil, subprocess, tempfile
from googleapiclient.http import MediaIoBaseDownload

DRIVE_PROJECT_ID = "1iCC4k4HgIxnz2RnyIY_qt7hxHSLoEVJB"
REPO_URL = "https://github.com/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications.git"
STUDENT_PATH = Path("students/hamza_arab")
WORK_DIR = Path("/content/oss4008_git_publish")
SIZE_LIMIT = 49_000_000

FOLDER_MAP = {
    "01_Notebooks": "code/notebooks",
    "02_App": "code/app",
    "screenshots": "screenshots",
}
ALLOWED_EXTENSIONS = {".py", ".ipynb", ".md", ".txt", ".json", ".geojson", ".csv", ".yml", ".yaml", ".png", ".jpg", ".jpeg", ".svg", ".js"}

def drive_children(folder_id):
    all_files = []
    token = None
    while True:
        resp = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id,name,mimeType,size)",
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        all_files.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            return all_files

def download_drive_file(item, target):
    if int(item.get("size", "0")) > SIZE_LIMIT:
        print("SKIP oversized file:", item["name"])
        return False
    if target.suffix.lower() not in ALLOWED_EXTENSIONS:
        print("SKIP non-source file:", item["name"])
        return False
    if item["name"] in {"legacy_requirements.txt"}:
        return False
    request = drive.files().get_media(fileId=item["id"])
    content = BytesIO()
    downloader = MediaIoBaseDownload(content, request)
    finished = False
    while not finished:
        _, finished = downloader.next_chunk()
    payload = content.getvalue()
    if target.suffix.lower() == ".ipynb":
        notebook = json.loads(payload.decode("utf-8"))
        for cell in notebook.get("cells", []):
            if cell.get("cell_type") == "code":
                cell["execution_count"] = None
                cell["outputs"] = []
            # Colab execution metadata can include account identifiers and output references.
            cell["metadata"] = {}
        notebook["metadata"] = {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}}
        payload = json.dumps(notebook, indent=1, ensure_ascii=False).encode("utf-8")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
    return True

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
repo_dir = WORK_DIR / "repository"
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
student_dir = repo_dir / STUDENT_PATH
student_dir.mkdir(parents=True, exist_ok=True)
# The user requested a streamlined repository. Remove ONLY outdated content
# inside the user's student folder, never any other student's files.
for subdir in ("code/notebooks", "code/app", "docs", "data", "results", "screenshots"):
    obsolete = student_dir / subdir
    if obsolete.is_dir():
        shutil.rmtree(obsolete)

folder_items = drive_children(DRIVE_PROJECT_ID)
written = []
for item in folder_items:
    if item["mimeType"] == "application/vnd.google-apps.folder" and item["name"] in FOLDER_MAP:
        destination = student_dir / FOLDER_MAP[item["name"]]
        # No existing destination contents are deleted.
        for source in drive_children(item["id"]):
            if source["mimeType"] == "application/vnd.google-apps.folder":
                print("SKIP nested folder (review before committing):", source["name"])
                continue
            target = destination / source["name"]
            if download_drive_file(source, target):
                written.append(str(target.relative_to(repo_dir)))
    elif item["name"] == "README.md":
        target = student_dir / "README.md"
        if download_drive_file(item, target):
            written.append(str(target.relative_to(repo_dir)))


if not written:
    raise RuntimeError("No project files could be copied from Google Drive. Do not push.")
print(f"Prepared {len(written)} project files:")
print("\n".join(written))

# Refuse to stage source that contains patterns typical of credentials.
# Match actual token formats and credential assignments, not ordinary words like "token".
SECRET_PATTERNS = [
    ("GitHub token", re.compile(r"(?:github_pat_|gh[pousr]_|ghu_)[A-Za-z0-9_]{12,}")),
    ("Google API key", re.compile(r"AIza[0-9A-Za-z_\-]{25,}")),
    ("AWS access key", re.compile(r"AKIA[0-9A-Z]{16}")),
    ("Private key block", re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----")),
    ("OAuth credential value", re.compile(r'(?i)["\\\'](?:client_secret|refresh_token|access_token|private_key)["\\\']\s*:\s*["\\\'][^"\\\']{8,}')),
    ("Bearer token", re.compile(r"(?i)Bearer\s+[A-Za-z0-9_\-.]{24,}")),
]
alerts = []
for rel in written:
    file = repo_dir / rel
    if file.suffix.lower() not in {".py", ".ipynb", ".md", ".txt", ".json", ".geojson", ".yml", ".yaml", ".csv"}:
        continue
    content = file.read_text(encoding="utf-8", errors="replace")
    for label, pattern in SECRET_PATTERNS:
        if pattern.search(content):
            alerts.append((rel, label))
if alerts:
    for rel, label in alerts:
        print("BLOCKED:", rel, "contains a possible", label)
    raise RuntimeError("Possible secrets detected. Review the files before pushing; no commit was created.")
print("Credential-pattern scan passed. This is not a substitute for reviewing files.")

subprocess.run(["git", "-C", str(repo_dir), "add", "--", str(STUDENT_PATH)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "status", "--short"], check=True)

In [ ]:
from getpass import getpass
from pathlib import Path
import os, stat, subprocess

# Publish only your student directory, following the TA's pull/rebase/push workflow.
# Run the Google Drive authentication and file-preparation cells above first.
repo_dir = Path('/content/oss4008_git_publish/repository')
student_path = 'students/hamza_arab'
if not (repo_dir / '.git').is_dir():
    raise RuntimeError('Repository missing. Run the preparation cell above first.')

def git(*args, env=None):
    return subprocess.run(['git', '-C', str(repo_dir), *args], env=env, check=True)

github_username = input('GitHub username: ').strip()
github_email = input('GitHub email for commits: ').strip()
github_token = getpass('GitHub personal access token (hidden): ').strip()
if not all((github_username, github_email, github_token)):
    raise ValueError('Username, email and personal access token are required.')

# Never insert the token in the remote URL or in code/notebook output.
helper = Path('/content/oss4008_git_askpass.sh')
helper.write_text('''#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "$OSS_GITHUB_USERNAME" ;;
  *Password*) printf '%s\\n' "$OSS_GITHUB_TOKEN" ;;
esac
''', encoding='utf-8')
helper.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
env = os.environ.copy()
env.update({
    'GIT_ASKPASS': str(helper),
    'GIT_TERMINAL_PROMPT': '0',
    'OSS_GITHUB_USERNAME': github_username,
    'OSS_GITHUB_TOKEN': github_token,
})

try:
    git('config', 'user.name', github_username)
    git('config', 'user.email', github_email)
    git('add', '--', student_path)

    staged = subprocess.run(
        ['git', '-C', str(repo_dir), 'diff', '--cached', '--quiet'],
        check=False
    )
    if staged.returncode == 1:
        git('commit', '-m', 'Update OSS4008 hospital mapping project progress')
    elif staged.returncode == 0:
        print('No new changes to commit; checking the remote anyway.')
    else:
        raise RuntimeError('Could not inspect staged changes.')

    # If someone else updated main since the clone, replay our commit on top.
    # This is the TA's required synchronization step.
    git('pull', '--rebase', 'origin', 'main', env=env)
    git('push', 'origin', 'HEAD:main', env=env)
    print('SUCCESS: Push completed.')
    print('https://github.com/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/tree/main/students/hamza_arab')
except subprocess.CalledProcessError as err:
    print('Publishing failed at:', ' '.join(err.cmd[:4]))
    print('Check that the repository invitation was accepted and your token has repo write access.')
    print('If Git reports a rebase conflict, resolve it before retrying; do not force-push.')
    raise
finally:
    helper.unlink(missing_ok=True)
    env.pop('OSS_GITHUB_TOKEN', None)
    github_token = ''
